# Per-Technique Tables (paper Tables 6–7)

Re-scores every technique × iteration for both models (GPT-5 and GPT-4o, t = 0) with one shared
set of metric functions, and prints the per-technique averages reported in the paper:

- printed below as **“Table 3”** → **paper Table 6** (Step 1 outputs);
- printed below as **“Table 4”** → **paper Table 7** (Steps 2–4 outputs).

It also runs the prompt-sensitivity analysis (range / SD across techniques) quoted in §4.1.2 and
saves the per-conversation detail to `results/per_technique_eval_detail.csv` — the file from which
the shipped `Step2_Score.csv` tables are derived.

**Requires the raw outputs** (`results/<model>/raw/`, git-ignored).

In [1]:
import json, os, glob, numpy as np, pandas as pd
from collections import defaultdict

# ── Configuration ──
DATA_DIR = os.path.join('.', 'data')
GOLD_DIR = os.path.join(DATA_DIR, 'gold')

FOURSTEP_DIRS = {
    'GPT-5':  'results/gpt5/raw',
    'GPT-4o': 'results/gpt4o/raw',
}

STEP1_TECHS = ['ND', 'ZS', 'CoT']
STEP234_TECHS = ['CoT', 'SR', 'PD', 'MoRE']
ITERATIONS = range(1, 6)

print('Configuration loaded')
print(f'Gold dir exists: {os.path.isdir(GOLD_DIR)}')
for label, d in FOURSTEP_DIRS.items():
    print(f'{label} dir exists: {os.path.isdir(d)}')

Configuration loaded
Gold dir exists: True
GPT-5 dir exists: True
GPT-4o dir exists: True


## Evaluation functions (paper Eq. 1–6)

Step 1.1 set-F1 · Step 1.2 (participant, label)-pair F1 · Step 2 Mentioned-pair F1 ·
Step 3 cell-level F1 · Step 4 Positive-F1 over the integrated factor table. Steps 2–4 are
scope-filtered by the pipeline's Step-1 best output (P*, R*).

In [2]:
def normalize_name(x):
    return str(x).strip().lower()

def f1_from_counts(tp, fp, fn):
    """Paper Eq.1: F1 from TP, FP, FN counts"""
    if tp == 0:
        return 0.0
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    return 2 * prec * rec / (prec + rec) if (prec + rec) > 0 else 0.0

def set_f1(gold_list, pred_list):
    """Step 1.1: Set-based F1 for participants/restaurants"""
    if not gold_list and not pred_list: return 1.0
    if not gold_list or not pred_list: return 0.0
    gold_set = {str(x).strip() for x in gold_list}
    pred_set = {str(x).strip() for x in pred_list}
    tp = len(gold_set & pred_set)
    return f1_from_counts(tp, len(pred_set - gold_set), len(gold_set - pred_set))

def table_f1(gold_table, pred_table, key_columns):
    """Step 1.2: Row-level F1 for suggestion/response tables
    Matches rows by key_columns (e.g., ['participant', 'suggestion_type']).
    Each unique combination of key values is one element in S."""
    if not gold_table and not pred_table: return 1.0
    if not gold_table or not pred_table: return 0.0
    def make_key(row):
        return ' '.join(str(row.get(c, '')).strip() for c in key_columns)
    gold_keys = set(make_key(r) for r in gold_table)
    pred_keys = set(make_key(r) for r in pred_table)
    tp = len(gold_keys & pred_keys)
    return f1_from_counts(tp, len(pred_keys - gold_keys), len(gold_keys - pred_keys))

def mentioned_f1(gold_table, pred_table, scope_restaurants):
    """Step 2: F1 on (participant, restaurant) pairs where mention='Mentioned'
    Scope-filtered by Step 1 restaurants."""
    scope_r = {normalize_name(r) for r in scope_restaurants}
    def extract(table):
        pairs = set()
        for row in table:
            if row.get('mention', '') == 'Mentioned':
                r = normalize_name(row.get('restaurant', ''))
                if r in scope_r:
                    pairs.add((row['participant'], r))
        return pairs
    gp, pp = extract(gold_table), extract(pred_table)
    if not gp and not pp: return 1.0
    tp = len(gp & pp)
    return f1_from_counts(tp, len(pp - gp), len(gp - pp))

def perception_micro_f1(gold_table, pred_table, scope_p, scope_r):
    """Step 3: Triplet micro F1 with scope filtering
    Compares (participant, restaurant) -> perception value."""
    scope_r_lower = {normalize_name(r) for r in scope_r}
    gold_map, pred_map = {}, {}
    for g in gold_table:
        if g['participant'] in scope_p and normalize_name(g['restaurant']) in scope_r_lower:
            gold_map[(g['participant'], normalize_name(g['restaurant']))] = g.get('perception', g.get('sentiment', '')).strip()
    for p in pred_table:
        if p['participant'] in scope_p and normalize_name(p['restaurant']) in scope_r_lower:
            pred_map[(p['participant'], normalize_name(p['restaurant']))] = p.get('sentiment', p.get('perception', '')).strip()
    tp = fp = fn = 0
    for key, gv in gold_map.items():
        if key in pred_map:
            if gv == pred_map[key]: tp += 1
            else: fn += 1
        else: fn += 1
    for key in pred_map:
        if key not in gold_map: fp += 1
    return f1_from_counts(tp, fp, fn)

def parse_factors(s):
    """Factor string -> set. 'A1,A2' -> {'A1','A2'}, 'None' -> set()"""
    if not s or str(s).strip().lower() == 'none': return set()
    return {f.strip() for f in str(s).split(',') if f.strip() and f.strip().lower() != 'none'}

def integrate_tables(pref_tbl, cons_tbl):
    """Merge preference + constraint tables into unified Interpretation Table."""
    union_map = {}
    for row in pref_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('preferences', 'None')))
    for row in cons_tbl:
        key = (row['participant'], normalize_name(row['restaurant']))
        union_map.setdefault(key, set()).update(parse_factors(row.get('constraints', 'None')))
    return union_map

def interpretation_positive_f1(gold_pref, gold_cons, pred_pref, pred_cons, scope_p, scope_r):
    """Step 4: Positive-F1 on integrated Interpretation Table (paper Eq.5-6)
    1. Scope-filter by Step 1 participants and restaurants
    2. Integrate preferences + constraints per (participant, restaurant)
    3. D_positive = cells where gold has at least one factor
    4. Positive-F1 = mean of per-cell F1 over D_positive"""
    scope_r_lower = {normalize_name(r) for r in scope_r}
    def filt(table):
        return [t for t in table
                if t['participant'] in scope_p
                and normalize_name(t['restaurant']) in scope_r_lower]
    gold_int = integrate_tables(filt(gold_pref), filt(gold_cons))
    pred_int = integrate_tables(filt(pred_pref), filt(pred_cons))
    pos_sum = pos_cnt = 0
    for key, gold_set in gold_int.items():
        pred_set = pred_int.get(key, set())
        if gold_set:
            inter = len(gold_set & pred_set)
            prec = inter / len(pred_set) if pred_set else 0.0
            rec  = inter / len(gold_set) if gold_set else 0.0
            f1 = 2 * prec * rec / (prec + rec) if (prec + rec) else 0.0
            pos_sum += f1
            pos_cnt += 1
    return pos_sum / pos_cnt if pos_cnt else 0.0

print('All evaluation functions defined')

All evaluation functions defined


## Data Loading

In [3]:
def load_gold(log_name):
    """Load ground truth for all steps"""
    gold_path = os.path.join(GOLD_DIR, log_name)
    gold = {}
    for key, fname in [('step1_1', 'step1_1_gold.json'), ('step1_2', 'step1_2_gold.json'),
                       ('step2', 'step2_gold.json'), ('step3', 'step3_gold.json'),
                       ('step4', 'step4_gold.json')]:
        fpath = os.path.join(gold_path, fname)
        if os.path.exists(fpath):
            with open(fpath, 'r', encoding='utf-8') as f:
                gold[key] = json.load(f)
    return gold

print('Data loading function defined')

Data loading function defined


## Run Per-Technique Evaluation

For each model × conversation × technique × iteration:
- **Step 1**: Load `Step1_{tech}_normalized_{it}.json`, compute 5 metrics
- **Steps 2-4**: Load `{log}_Step{k}_{tech}_{it}.json`, compute F1/Pos-F1
- **Scope**: From `final_results.json` Step 1 output (participants, restaurants)

In [4]:
all_rows = []

for model_label, fourstep_dir in FOURSTEP_DIRS.items():
    if not os.path.isdir(fourstep_dir):
        print(f'WARNING: {fourstep_dir} not found, skipping {model_label}')
        continue
    logs = sorted([d for d in os.listdir(fourstep_dir)
                   if d.endswith('_log') and os.path.isdir(os.path.join(fourstep_dir, d))])
    print(f'{model_label}: {len(logs)} conversations')

    for log_name in logs:
        gold = load_gold(log_name)
        if not gold: continue

        # Scope from Step 1 best output
        final_path = os.path.join(fourstep_dir, log_name, f'{log_name}_final_results.json')
        if not os.path.exists(final_path): continue
        with open(final_path) as f:
            final = json.load(f)
        scope_p = set(final['step1']['participants'])
        scope_r = set(final['step1']['restaurant_brands'])

        # Step 1: ND, ZS, CoT
        for tech in STEP1_TECHS:
            scores = {m: [] for m in ['F1_Participants','F1_Restaurants','Final_Restaurant_Match','F1_Suggestion','F1_Response']}
            for it in ITERATIONS:
                fpath = os.path.join(fourstep_dir, log_name, f'Step1_{tech}_normalized_{it}.json')
                if not os.path.exists(fpath): continue
                with open(fpath) as f: pred = json.load(f)
                g11 = gold.get('step1_1', {})
                g12 = gold.get('step1_2', {})
                scores['F1_Participants'].append(set_f1(g11.get('participants',[]), pred.get('participants',[])))
                scores['F1_Restaurants'].append(set_f1(g11.get('restaurant_brands',[]), pred.get('restaurant_brands',[])))
                scores['Final_Restaurant_Match'].append(
                    1.0 if g11.get('final_restaurant','').strip() == pred.get('final_restaurant','').strip() else 0.0)
                scores['F1_Suggestion'].append(table_f1(g12.get('suggestion_table',[]), pred.get('suggestion_table',[]),
                                                        ['participant','suggestion_type']))
                scores['F1_Response'].append(table_f1(g12.get('response_table',[]), pred.get('response_table',[]),
                                                      ['participant','response_type']))
            if scores['F1_Participants']:
                row = {'model': model_label, 'step': 'Step1', 'technique': tech, 'log': log_name}
                for m in scores: row[m] = np.mean(scores[m])
                all_rows.append(row)

        # Steps 2-4: CoT, SR, PD, MoRE
        for tech in STEP234_TECHS:
            s2, s3, s4 = [], [], []
            for it in ITERATIONS:
                fp2 = os.path.join(fourstep_dir, log_name, f'{log_name}_Step2_{tech}_{it}.json')
                if os.path.exists(fp2):
                    with open(fp2) as f: d2 = json.load(f)
                    s2.append(mentioned_f1(gold.get('step2',{}).get('mentioned_table',[]),
                                           d2.get('Result',{}).get('mentioned_table',[]), scope_r))
                fp3 = os.path.join(fourstep_dir, log_name, f'{log_name}_Step3_{tech}_{it}.json')
                if os.path.exists(fp3):
                    with open(fp3) as f: d3 = json.load(f)
                    g3 = gold.get('step3',{})
                    g3t = g3.get('perception_table', g3.get('sentiment_table',[]))
                    s3.append(perception_micro_f1(g3t, d3.get('Result',{}).get('sentiment_table',[]), scope_p, scope_r))
                fp4 = os.path.join(fourstep_dir, log_name, f'{log_name}_Step4_{tech}_{it}.json')
                if os.path.exists(fp4):
                    with open(fp4) as f: d4 = json.load(f)
                    g4 = gold.get('step4',{})
                    s4.append(interpretation_positive_f1(
                        g4.get('preference_table',[]), g4.get('constraint_table',[]),
                        d4.get('Result',{}).get('preference_table',[]),
                        d4.get('Result',{}).get('constraint_table',[]), scope_p, scope_r))

            row = {'model': model_label, 'step': 'Steps2-4', 'technique': tech, 'log': log_name}
            row['F1_Mentioned'] = np.mean(s2) if s2 else np.nan
            row['F1_PerceptionTable'] = np.mean(s3) if s3 else np.nan
            row['F1_InterpretationTable'] = np.mean(s4) if s4 else np.nan
            all_rows.append(row)

df_all = pd.DataFrame(all_rows)
print(f'\nTotal rows: {len(df_all)}')
df_all.head()

GPT-5: 47 conversations
GPT-4o: 47 conversations

Total rows: 658


,model,step,technique,log,F1_Participants,F1_Restaurants,Final_Restaurant_Match,F1_Suggestion,F1_Response,F1_Mentioned,F1_PerceptionTable,F1_InterpretationTable
0,GPT-5,Step1,ND,10_log,1.0,1.0,1.0,0.733333,0.466667,NaN,NaN,NaN
1,GPT-5,Step1,ZS,10_log,1.0,1.0,1.0,0.666667,0.400000,NaN,NaN,NaN
2,GPT-5,Step1,CoT,10_log,1.0,1.0,1.0,0.600000,0.600000,NaN,NaN,NaN
3,GPT-5,Steps2-4,CoT,10_log,NaN,NaN,NaN,NaN,NaN,1.0,0.775172,0.452667
4,GPT-5,Steps2-4,SR,10_log,NaN,NaN,NaN,NaN,NaN,1.0,0.749754,0.448000


## Step 1 per-technique averages — paper **Table 6** (printed as “Table 3”)

In [5]:
step1 = df_all[df_all['step'] == 'Step1']
step1_metrics = ['F1_Participants', 'F1_Restaurants', 'Final_Restaurant_Match', 'F1_Suggestion', 'F1_Response']
metric_labels = ['Participant Lists', 'Restaurant Lists', 'Chosen Restaurant', 'Suggestion Lists', 'Response Lists']

results = []
for model in ['GPT-5', 'GPT-4o']:
    sub = step1[step1['model'] == model]
    for metric, label in zip(step1_metrics, metric_labels):
        row = {'Model': model, 'Output': label}
        for tech in STEP1_TECHS:
            t = sub[sub['technique'] == tech]
            row[f'{tech}_Ave'] = round(t[metric].mean(), 2)
            row[f'{tech}_Std'] = round(t[metric].std(), 2)
        ave_m = sub.groupby('log')[metric].mean().mean()
        ave_s = sub.groupby('log')[metric].mean().std()
        row['Ave_Ave'] = round(ave_m, 2)
        row['Ave_Std'] = round(ave_s, 2)
        results.append(row)

table3 = pd.DataFrame(results)
print('Table 3: Step 1 Per-Technique F1 Scores')
print('=' * 100)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n{model}:')
    sub = table3[table3['Model'] == model]
    for _, row in sub.iterrows():
        line = f"  {row['Output']:<20}"
        for tech in STEP1_TECHS:
            line += f"  {row[f'{tech}_Ave']:.2f} ({row[f'{tech}_Std']:.2f})"
        line += f"  {row['Ave_Ave']:.2f} ({row['Ave_Std']:.2f})"
        print(line)

Table 3: Step 1 Per-Technique F1 Scores

GPT-5:
  Participant Lists     1.00 (0.00)  1.00 (0.00)  1.00 (0.00)  1.00 (0.00)
  Restaurant Lists      1.00 (0.02)  1.00 (0.01)  1.00 (0.02)  1.00 (0.02)
  Chosen Restaurant     0.92 (0.23)  0.96 (0.12)  0.95 (0.19)  0.95 (0.17)
  Suggestion Lists      0.69 (0.23)  0.67 (0.23)  0.65 (0.23)  0.67 (0.22)
  Response Lists        0.38 (0.31)  0.42 (0.29)  0.44 (0.26)  0.41 (0.28)

GPT-4o:
  Participant Lists     1.00 (0.00)  1.00 (0.00)  1.00 (0.00)  1.00 (0.00)
  Restaurant Lists      0.98 (0.07)  0.97 (0.09)  0.97 (0.10)  0.97 (0.08)
  Chosen Restaurant     0.90 (0.30)  0.90 (0.29)  0.94 (0.23)  0.91 (0.26)
  Suggestion Lists      0.65 (0.21)  0.70 (0.21)  0.68 (0.20)  0.68 (0.19)
  Response Lists        0.26 (0.34)  0.28 (0.32)  0.29 (0.30)  0.27 (0.31)


## Steps 2–4 per-technique averages — paper **Table 7** (printed as “Table 4”)

In [6]:
step234 = df_all[df_all['step'] == 'Steps2-4']
step234_metrics = ['F1_Mentioned', 'F1_PerceptionTable', 'F1_InterpretationTable']
step234_labels = ['Mentioned Table (Step 2)', 'Perception Table (Step 3)', 'Interpretation Table (Step 4)']

results4 = []
for model in ['GPT-5', 'GPT-4o']:
    sub = step234[step234['model'] == model]
    for metric, label in zip(step234_metrics, step234_labels):
        row = {'Model': model, 'Output': label}
        for tech in STEP234_TECHS:
            t = sub[sub['technique'] == tech]
            row[f'{tech}_Ave'] = round(t[metric].mean(), 2)
            row[f'{tech}_Std'] = round(t[metric].std(), 2)
        ave_m = sub.groupby('log')[metric].mean().mean()
        ave_s = sub.groupby('log')[metric].mean().std()
        row['Ave_Ave'] = round(ave_m, 2)
        row['Ave_Std'] = round(ave_s, 2)
        results4.append(row)

table4 = pd.DataFrame(results4)
print('Table 4: Steps 2-4 Per-Technique F1 Scores')
print('=' * 100)
for model in ['GPT-5', 'GPT-4o']:
    print(f'\n{model}:')
    sub = table4[table4['Model'] == model]
    for _, row in sub.iterrows():
        line = f"  {row['Output']:<30}"
        for tech in STEP234_TECHS:
            line += f"  {row[f'{tech}_Ave']:.2f} ({row[f'{tech}_Std']:.2f})"
        line += f"  {row['Ave_Ave']:.2f} ({row['Ave_Std']:.2f})"
        print(line)

Table 4: Steps 2-4 Per-Technique F1 Scores

GPT-5:
  Mentioned Table (Step 2)        0.97 (0.07)  0.98 (0.06)  0.97 (0.07)  0.97 (0.07)  0.97 (0.07)
  Perception Table (Step 3)       0.84 (0.10)  0.83 (0.10)  0.84 (0.10)  0.83 (0.11)  0.84 (0.10)
  Interpretation Table (Step 4)   0.37 (0.21)  0.38 (0.21)  0.40 (0.20)  0.39 (0.20)  0.38 (0.20)

GPT-4o:
  Mentioned Table (Step 2)        0.88 (0.17)  0.88 (0.15)  0.92 (0.14)  0.90 (0.14)  0.90 (0.14)
  Perception Table (Step 3)       0.77 (0.20)  0.75 (0.21)  0.65 (0.28)  0.79 (0.19)  0.74 (0.19)
  Interpretation Table (Step 4)   0.37 (0.25)  0.37 (0.25)  0.37 (0.25)  0.37 (0.25)  0.37 (0.25)


## Prompt sensitivity — range and SD across techniques (paper §4.1.2)

In [7]:
print('Prompt Sensitivity: Range and SD of per-technique means')
print('=' * 100)

for model in ['GPT-5', 'GPT-4o']:
    print(f'\n{model}:')
    sub = step234[step234['model'] == model]
    for metric, label in [('F1_Mentioned', 'Step 2 Mentioned'),
                           ('F1_PerceptionTable', 'Step 3 Perception'),
                           ('F1_InterpretationTable', 'Step 4 Interpretation')]:
        tech_avgs = []
        for tech in STEP234_TECHS:
            t = sub[sub['technique'] == tech]
            tech_avgs.append(t[metric].mean())
        rng = max(tech_avgs) - min(tech_avgs)
        sd = np.std(tech_avgs)
        detail = ', '.join(f'{t}={v:.3f}' for t, v in zip(STEP234_TECHS, tech_avgs))
        print(f'  {label:<25} Range={rng:.3f}  SD={sd:.3f}  [{detail}]')

Prompt Sensitivity: Range and SD of per-technique means

GPT-5:
  Step 2 Mentioned          Range=0.009  SD=0.003  [CoT=0.971, SR=0.977, PD=0.968, MoRE=0.974]
  Step 3 Perception         Range=0.013  SD=0.005  [CoT=0.843, SR=0.830, PD=0.839, MoRE=0.834]
  Step 4 Interpretation     Range=0.026  SD=0.010  [CoT=0.370, SR=0.379, PD=0.396, MoRE=0.389]

GPT-4o:
  Step 2 Mentioned          Range=0.045  SD=0.018  [CoT=0.877, SR=0.881, PD=0.922, MoRE=0.901]
  Step 3 Perception         Range=0.136  SD=0.052  [CoT=0.768, SR=0.748, PD=0.650, MoRE=0.786]
  Step 4 Interpretation     Range=0.001  SD=0.001  [CoT=0.366, SR=0.367, PD=0.367, MoRE=0.366]


## Save Results

In [8]:
df_all.to_csv('results/per_technique_eval_detail.csv', index=False)
print('Saved: results/per_technique_eval_detail.csv')
print(f'Shape: {df_all.shape}')

Saved: per_technique_eval_detail.csv
Shape: (658, 12)
